In [1]:
import numpy as np
import pandas as pd
import warnings

%load_ext autoreload
%autoreload 2

import utils.transform as transform
import utils.psychometrics as psychometrics

In [2]:
warnings.filterwarnings(
    "ignore",
    message=".*force_all_finite.*",
    category=FutureWarning
)

BASE_PATH = "../final_data/raw/data_basis"

Master data: `'master.xlsx'` <br>
Age: `'age.xlsx'` <br>
SES: `'ses.xlsx'` <br>
CES_D: `'cesd.xlsx'` <br>
GAD7: `'gad7.xlsx'` <br>
SWLS: `'swls.xlsx'` <br>
IDS: `'ids.xlsx'` <br>
GDS15: `'gds15.xlsx'`

# Master Data

In [3]:
# --------------------------------------------------
# Read participant information
# --------------------------------------------------
data = pd.read_excel(
    f"{BASE_PATH}/master.xlsx",
    index_col="TEILNEHMER_SIC"
)

# --------------------------------------------------
# Sex – LIFE-Adult: 1 = male, 2 = female
# --------------------------------------------------
sex_map = {1: 0, 2: 1}  # Recode: 0 = male, 1 = female

data["TEILNEHMER_GESCHLECHT"] = data["TEILNEHMER_GESCHLECHT"].map(sex_map)

# --------------------------------------------------
# Birth year/month
# --------------------------------------------------
data["TEILNEHMER_GEB_JJJJMM"] = pd.to_datetime(
    data["TEILNEHMER_GEB_JJJJMM"],
    format="%Y%m",
    errors="coerce"
)

data = data[["TEILNEHMER_GESCHLECHT", "TEILNEHMER_GEB_JJJJMM"]]

# --------------------------------------------------
# Age at baseline examination
# --------------------------------------------------
age = pd.read_excel(
    f"{BASE_PATH}/age.xlsx",
    index_col="SIC"
)

age = age[["ADULT_PROB_AGE"]].copy()

age["AGE_YEARS"] = np.floor(age["ADULT_PROB_AGE"]).astype("Int64")

age["AGE_MONTHS"] = np.round(
    (age["ADULT_PROB_AGE"] - age["AGE_YEARS"]) * 12
).astype("Int64")

data = data.join(age, how="left")

# --------------------------------------------------
# Calculate examination date
# --------------------------------------------------
def estimate_examination_date(row):
    """Estimate baseline examination date from birth year/month and age."""
    birth = row["TEILNEHMER_GEB_JJJJMM"]
    years = row["AGE_YEARS"]
    months = row["AGE_MONTHS"]
    if pd.isna(birth) or pd.isna(years) or pd.isna(months):
        return pd.NaT
    return birth + pd.DateOffset(years=int(years), months=int(months))

data["BASIS_EXAMINATION_DATE"] = data.apply(estimate_examination_date, axis=1)

# --------------------------------------------------
# Examination year
# --------------------------------------------------
data["BASIS_EXAMINATION_YEAR"] = data["BASIS_EXAMINATION_DATE"].dt.year

# --------------------------------------------------
# Examination season
# --------------------------------------------------
SEASON_MAP = {             # month-season mapping dictionary
    2: 1, 3: 1, 4: 1,      # Season 1 = Spring (Feb-Apr)
    5: 2, 6: 2, 7: 2,      # Season 2 = Summer (May-Jul)
    8: 3, 9: 3, 10: 3,     # Season 3 = Autumn (Aug-Oct)
    11: 4, 12: 4, 1: 4     # Season 4 = Winter (Nov-Jan)
}

data["BASIS_EXAMINATION_SEASON"] = (
    data["BASIS_EXAMINATION_DATE"]
    .dt.month
    .map(SEASON_MAP)
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "TEILNEHMER_GESCHLECHT",
    "ADULT_PROB_AGE",
    "BASIS_EXAMINATION_YEAR",
    "BASIS_EXAMINATION_SEASON",
]

df_basis = data[columns].copy()

df_basis

,TEILNEHMER_GESCHLECHT,ADULT_PROB_AGE,BASIS_EXAMINATION_YEAR,BASIS_EXAMINATION_SEASON
TEILNEHMER_SIC,,,,
D60B916F4D,1,61.08,2013,2
F9BD339F59,1,42.11,2014,3
4423166C4A,1,41.80,2014,4
128A8A2817,1,61.46,2011,4
7E64C9A93D,0,63.50,2012,3
...,...,...,...,...
E5B1AC509A,1,56.71,2012,4
9FA0ABBA6D,1,34.69,2011,3
1A8DDEA337,0,48.15,2012,2


# SES
## Socioeconomical Status

In [4]:
# --------------------------------------------------
# Read SES data
# --------------------------------------------------
df_ses = pd.read_excel(
    f"{BASE_PATH}/ses.xlsx",
    index_col="SIC"
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "SES2_EDAT",
    "SES2_SES5",
    "SES2_SES3",
]

df_ses = df_ses[columns]

display(df_ses)

,SES2_EDAT,SES2_SES5,SES2_SES3
SIC,,,
23FB02CD4F,2014-04-02,1.0,1.0
E038457A98,2014-04-02,2.0,2.0
6AEC0808FF,2014-03-28,2.0,2.0
B33B567472,2014-03-24,2.0,2.0
CD1BAA9A9D,2014-03-21,3.0,2.0
...,...,...,...
6492CD7D15,2011-08-15,1.0,1.0
13B1032389,2011-08-15,4.0,2.0
0DD2570998,2011-08-15,4.0,2.0


# CES-D
## Center for Epidemiologic Studies Depression Scale

In [5]:
# --------------------------------------------------
# Read CES-D data
# --------------------------------------------------
df_cesd = pd.read_excel(
    f"{BASE_PATH}/cesd.xlsx",
    index_col="SIC"
)
display(df_cesd.info())

# --------------------------------------------------
# CES-D items
# --------------------------------------------------
subset = [f"CES_D_{i}" for i in range(1, 21)] # _1:_20

# --------------------------------------------------
# Person-mean imputation
# Impute missing values using each participant's mean
# score when ≤20% of items (≤4 of 20) are missing
# --------------------------------------------------
df_cesd = psychometrics.person_mean_imputation(
    df_cesd,
    subset_columns=subset,
    threshold=1/5,
)

assert not df_cesd[subset].isna().any().any()

# --------------------------------------------------
# Internal consistency
# --------------------------------------------------
reliability = psychometrics.reliability_test(df_cesd, subset)

# --------------------------------------------------
# Compute CES-D total score (range: 0–60)
# Higher scores indicate greater depressive symptoms
# --------------------------------------------------
df_cesd["CES_D_SUM"] = df_cesd[subset].sum(axis=1)

# --------------------------------------------------
# Depression severity
#
# 0:  0–22 → No clinically relevant depressive symptoms
# 1: 23–32 → Elevated depressive symptoms / at risk
# 2: 33–60 → Probable clinical depression
# --------------------------------------------------
df_cesd["CESD_depression_severity"] = transform.categorize(
    df_cesd,
    "CES_D_SUM",
    bins=[0,23,33,np.inf],
    right=False,
)

# --------------------------------------------------
# Binary screening outcome
#
# 0: CES-D <  23 → No depression
# 1: CES-D >= 23 → Depressive symptoms (screen-positive)
# --------------------------------------------------
df_cesd["CESD_clinical_depression"] = transform.binarize(
    df_cesd,
    "CES_D_SUM",
    threshold=23,
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "CES_D_DATUM",
    "CES_D_SUM",
    "CESD_depression_severity",
    "CESD_clinical_depression",
]
df_cesd = df_cesd[columns]

display(df_cesd)

<class 'pandas.core.frame.DataFrame'>
Index: 9912 entries, 23FB02CD4F to F5D9FBE1FA
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   CES_D_DATUM         9912 non-null   datetime64[ns]
 1   GRUPPE              9912 non-null   object        
 2   CES_D_DQP_ID        9912 non-null   object        
 3   CES_D_UNTERSUCHER   7524 non-null   object        
 4   CES_D_DOKUID        0 non-null      float64       
 5   CES_D_BID           0 non-null      float64       
 6   CES_D_STARTZEIT     7537 non-null   datetime64[ns]
 7   CES_D_ENDZEIT       7537 non-null   datetime64[ns]
 8   CES_D_TECH_CHECKED  9912 non-null   int64         
 9   CES_D_CURATED       9912 non-null   int64         
 10  CES_D_CHECKED       9912 non-null   int64         
 11  CES_D_1             9790 non-null   float64       
 12  CES_D_2             9811 non-null   float64       
 13  CES_D_3             9764 non-null   fl

None


Missing Imputation
----------------------------------------
Of 9912 participants, 345 (3.5%) excluded for missing >20% of questionnaire items,
leaving a final sample of 9567 cases with 8868 complete and 699 imputed.

Internal consistency
----------------------------------------
Number of participants : 9,567
Number of items        : 20
McDonald's Omega       : 0.886
Cronbach's Alpha       : 0.867




,CES_D_DATUM,CES_D_SUM,CESD_depression_severity,CESD_clinical_depression
SIC,,,,
6AEC0808FF,2014-03-28 07:34:37,3.0,0,0
B33B567472,2014-03-24 00:00:00,23.0,1,1
CD1BAA9A9D,2014-03-21 00:00:00,14.0,0,0
9B34820501,2014-03-29 07:37:33,12.0,0,0
69B9B4A967,2014-04-02 07:15:36,14.0,0,0
...,...,...,...,...
6492CD7D15,2011-08-15 11:54:02,13.0,0,0
13B1032389,2011-08-15 10:07:26,5.0,0,0
0DD2570998,2011-08-15 12:45:53,3.0,0,0


# GAD-7

In [6]:
# --------------------------------------------------
# Read GAD-7 data
# --------------------------------------------------
df_gad7 = pd.read_excel(
    f"{BASE_PATH}/gad7.xlsx",
    index_col="SIC"
)
display(df_gad7.info())

# --------------------------------------------------
# GAD-7 items
# --------------------------------------------------
subset = [f"GAD7_GAD7_{i}" for i in range(1, 8)] # _1:_7

# --------------------------------------------------
# Person-mean imputation
# Impute missing values using each participant's mean
# score when at most one of the seven items is
# missing (≤14.3% missing).
# --------------------------------------------------
df_gad7 = psychometrics.person_mean_imputation(
    df_gad7,
    subset_columns=subset,
    threshold=1/7,
)

assert not df_gad7[subset].isna().any().any()

# --------------------------------------------------
# Internal consistency
# --------------------------------------------------
reliability = psychometrics.reliability_test(df_gad7, subset)
df_tmp = df_gad7[subset]

# --------------------------------------------------
# Compute GAD-7 total score (range: 0–21)
# Higher scores indicate greater anxiety severity
# --------------------------------------------------
df_gad7["GAD7_SUM"] = df_gad7[subset].sum(axis=1)

# --------------------------------------------------
# Anxiety severity 
#
# 0:  0– 4 → Minimal anxiety
# 1:  5– 9 → Mild anxiety
# 2: 10–14 → Moderate anxiety
# 3: 15–21 → Sever anxiety
# --------------------------------------------------
df_gad7["GAD7_anxiety_severity"] = transform.categorize(
    df_gad7,
    "GAD7_SUM",
    bins=[0,5,10,15,np.inf],
    right=False,
)

# --------------------------------------------------
# Binary screening outcome
#
# 0: GAD-7 <  10 → No clinically relevant anxiety
# 1: GAD-7 >= 10 → Clinically relevant anxiety (screen-positive)
# --------------------------------------------------
df_gad7["GAD7_anxiety"] = transform.binarize(
    df_gad7,
    "GAD7_SUM",
    threshold=10,
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "GAD7_DATUM",
    "GAD7_SUM",
    "GAD7_anxiety_severity",
    "GAD7_anxiety",
]
df_gad7 = df_gad7[columns]

display(df_gad7)

<class 'pandas.core.frame.DataFrame'>
Index: 9855 entries, 23FB02CD4F to F5D9FBE1FA
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   GAD7_DATUM         9855 non-null   datetime64[ns]
 1   GRUPPE             9855 non-null   object        
 2   GAD7_DQP_ID        9855 non-null   object        
 3   GAD7_UNTERSUCHER   7454 non-null   object        
 4   GAD7_DOKUID        0 non-null      float64       
 5   GAD7_BID           0 non-null      float64       
 6   GAD7_STARTZEIT     9855 non-null   datetime64[ns]
 7   GAD7_ENDZEIT       7466 non-null   datetime64[ns]
 8   GAD7_TECH_CHECKED  9855 non-null   int64         
 9   GAD7_CURATED       9855 non-null   int64         
 10  GAD7_CHECKED       9855 non-null   int64         
 11  GAD7_GAD7_1        9745 non-null   float64       
 12  GAD7_GAD7_2        9708 non-null   float64       
 13  GAD7_GAD7_3        9707 non-null   float64       
 14

None


Missing Imputation
----------------------------------------
Of 9855 participants, 129 (1.3%) excluded for missing >14% of questionnaire items,
leaving a final sample of 9726 cases with 9509 complete and 217 imputed.

Internal consistency
----------------------------------------
Number of participants : 9,726
Number of items        : 7
McDonald's Omega       : 0.864
Cronbach's Alpha       : 0.856




,GAD7_DATUM,GAD7_SUM,GAD7_anxiety_severity,GAD7_anxiety
SIC,,,,
23FB02CD4F,2014-04-02 00:00:00,3.0,0,0
6AEC0808FF,2014-03-28 07:34:37,0.0,0,0
B33B567472,2014-03-24 00:00:00,8.0,1,0
CD1BAA9A9D,2014-03-21 00:00:00,3.0,0,0
9B34820501,2014-03-29 07:37:33,0.0,0,0
...,...,...,...,...
C65D4C3F88,2011-08-15 09:58:12,5.0,1,0
6492CD7D15,2011-08-15 12:56:30,2.0,0,0
13B1032389,2011-08-15 11:38:59,1.0,0,0


# SWLS
## Satisfaction With Life Score

In [7]:
# --------------------------------------------------
# Read SWLS data
# --------------------------------------------------
df_swls = pd.read_excel(
    f"{BASE_PATH}/swls.xlsx",
    index_col="SIC"
)
display(df_swls.info())

# --------------------------------------------------
# SWLS items
#
# In the LIFE-Adult dataset, response categories are
# coded in the opposite direction of the original
# Satisfaction With Life Scale (SWLS). Therefore,
# all items are reverse scored so that higher scores
# consistently indicate greater life satisfaction.
# --------------------------------------------------
subset = [f"SWLS_SWLS_{i}" for i in range(1, 6)] # _1:_5

# Reverse score (1↔7, 2↔6, 3↔5, 4↔4)
df_swls[subset] = 8 - df_swls[subset]

# --------------------------------------------------
# Person-mean imputation
# Impute missing values using each participant's mean
# score when at most one of the five items is missing
# (≤20% missing)
# --------------------------------------------------
df_swls = psychometrics.person_mean_imputation(
    df_swls,
    subset_columns=subset,
    threshold=1/5,
)

assert not df_swls[subset].isna().any().any()

# --------------------------------------------------
# Internal consistency
# --------------------------------------------------
reliability = psychometrics.reliability_test(df_swls, subset)

# --------------------------------------------------
# Compute SWLS total score (range: 5–35)
# Higher scores indicate greater life satisfaction
# --------------------------------------------------
df_swls["SWLS_SUM"] = df_swls[subset].sum(axis=1)

# --------------------------------------------------
# Life satisfaction (Diener et al.)
#
#  5– 9 → Extremely dissatisfied
# 10–14 → Dissatisfied
# 15–19 → Slightly dissatisfied
# 20    → Neutral
# 21–25 → Slightly satisfied
# 26–30 → Satisfied
# 31–35 → Extremely satisfied
# --------------------------------------------------
df_swls["SWLS_satisfaction_grade"] = transform.categorize(
    df_swls,
    "SWLS_SUM",
    bins=[0,10,15,20,21,26,31,np.inf],
    right=False,
)

# --------------------------------------------------
# Binary life satisfaction
#
# 0: SWLS <  21 → Neutral or dissatisfied
# 1: SWLS >= 21 → Satisfied (slightly to extremely)
# --------------------------------------------------
df_swls["SWLS_satisfaction"] = transform.binarize(
    df_swls,
    "SWLS_SUM",
    threshold=21,
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "SWLS_DATUM",
    "SWLS_SUM",
    "SWLS_satisfaction_grade",
    "SWLS_satisfaction",
]
df_swls = df_swls[columns]

display(df_swls)

<class 'pandas.core.frame.DataFrame'>
Index: 9875 entries, 23FB02CD4F to F5D9FBE1FA
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   SWLS_DATUM         9875 non-null   datetime64[ns]
 1   GRUPPE             9875 non-null   object        
 2   SWLS_DQP_ID        9875 non-null   object        
 3   SWLS_UNTERSUCHER   7476 non-null   object        
 4   SWLS_DOKUID        0 non-null      float64       
 5   SWLS_BID           0 non-null      float64       
 6   SWLS_STARTZEIT     7488 non-null   datetime64[ns]
 7   SWLS_ENDZEIT       7488 non-null   datetime64[ns]
 8   SWLS_TECH_CHECKED  9875 non-null   int64         
 9   SWLS_CURATED       9875 non-null   int64         
 10  SWLS_CHECKED       9875 non-null   int64         
 11  SWLS_SWLS_1        9741 non-null   float64       
 12  SWLS_SWLS_2        9732 non-null   float64       
 13  SWLS_SWLS_3        9747 non-null   float64       
 14

None


Missing Imputation
----------------------------------------
Of 9875 participants, 127 (1.3%) excluded for missing >20% of questionnaire items,
leaving a final sample of 9748 cases with 9666 complete and 82 imputed.

Internal consistency
----------------------------------------
Number of participants : 9,748
Number of items        : 5
McDonald's Omega       : 0.901
Cronbach's Alpha       : 0.901




,SWLS_DATUM,SWLS_SUM,SWLS_satisfaction_grade,SWLS_satisfaction
SIC,,,,
23FB02CD4F,2014-04-02 00:00:00,20.0,3,0
E038457A98,2014-04-02 00:00:00,26.0,5,1
6AEC0808FF,2014-03-28 07:34:37,29.0,5,1
B33B567472,2014-03-24 00:00:00,25.0,4,1
CD1BAA9A9D,2014-03-21 00:00:00,29.0,5,1
...,...,...,...,...
6492CD7D15,2011-08-15 13:17:49,16.0,2,0
13B1032389,2011-08-15 11:47:06,34.0,6,1
0DD2570998,2011-08-15 13:24:43,31.0,6,1


# IDS-SR
## Inventory of Depressive Symptomatology - Self Report

In [8]:
# --------------------------------------------------
# Read IDS-SR data
# --------------------------------------------------
df_ids = pd.read_excel(
    f"{BASE_PATH}/ids.xlsx",
    index_col="SIC"
)
display(df_ids.info())

# IDS_I09A contains the scored response
df_ids["IDS_I09"] = df_ids[["IDS_I09A"]].copy()

# --------------------------------------------------
# IDS-SR items (30 items)
# --------------------------------------------------
subset = [f"IDS_I{i:02d}" for i in range(1, 31)] # _I01:_I30

# --------------------------------------------------
# Person-mean imputation
# Impute missing values using each participant's mean
# score when ≤20% of items (≤6 of 30) are missing
# --------------------------------------------------
df_ids = psychometrics.person_mean_imputation(
    df_ids,
    subset_columns=subset,
    threshold=1/5,
)

assert not df_ids[subset].isna().any().any()

# --------------------------------------------------
# Internal consistency
# --------------------------------------------------
reliability = psychometrics.reliability_test(df_ids, subset)

# --------------------------------------------------
# Collapse mutually exclusive symptom domains
# according to the IDS-SR scoring algorithm
# --------------------------------------------------
df_ids["appetite"] = df_ids[["IDS_I11", "IDS_I12"]].max(axis=1)
df_ids["weight"]   = df_ids[["IDS_I13", "IDS_I14"]].max(axis=1)

collapsed_items = ["IDS_I11", "IDS_I12", "IDS_I13", "IDS_I14"]

scored_items = [item for item in subset if item not in collapsed_items]
scored_items.extend(["appetite", "weight"])

# --------------------------------------------------
# Compute IDS-SR total score (range: 0–84)
# Higher scores indicate greater depressive symptoms
# --------------------------------------------------
df_ids["IDS_SUM"] = df_ids[scored_items].sum(axis=1)

# --------------------------------------------------
# Depression severity
#
# 0:  0–13 → No clinically relevant depressive symptoms
# 1: 14–25 → Mild depressive symptoms
# 2: 26–38 → Moderate depressive symptoms
# 3: 39–48 → Severe depressive symptoms
# 4: 49–84 → Very severe depressive symptoms
# --------------------------------------------------
df_ids["IDS_depression_severity"] = transform.categorize(
    df_ids,
    "IDS_SUM",
    bins=[0,14,26,39,49,np.inf],
    right=False,
)

# --------------------------------------------------
# Binary screening outcome
#
# 0: IDS-SR <  26 → No clinically significant depression
# 1: IDS-SR >= 26 → Moderate to very severe depressive symptoms
# --------------------------------------------------
df_ids["IDS_clinical_depression"] = transform.binarize(
    df_ids,
    "IDS_SUM",
    threshold=26,
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "IDS_EDAT",
    "IDS_SUM",
    "IDS_depression_severity",
    "IDS_clinical_depression",
]
df_ids = df_ids[columns]

display(df_ids)

<class 'pandas.core.frame.DataFrame'>
Index: 2991 entries, 1C0B901CC0 to 37AD8E09F4
Data columns (total 37 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   IDS_EDAT         2991 non-null   datetime64[ns]
 1   GRUPPE           2991 non-null   object        
 2   IDS_STARTZEIT    1620 non-null   datetime64[ns]
 3   IDS_ENDZEIT      1620 non-null   datetime64[ns]
 4   IDS_DQP_ID       2991 non-null   object        
 5   IDS_UNTERSUCHER  1620 non-null   object        
 6   IDS_I01          2978 non-null   float64       
 7   IDS_I02          2974 non-null   float64       
 8   IDS_I03          2955 non-null   float64       
 9   IDS_I04          2978 non-null   float64       
 10  IDS_I05          2974 non-null   float64       
 11  IDS_I06          2974 non-null   float64       
 12  IDS_I07          2979 non-null   float64       
 13  IDS_I08          2963 non-null   float64       
 14  IDS_I09A         2976 non-null

None


Missing Imputation
----------------------------------------
Of 2991 participants, 24 (0.8%) excluded for missing >20% of questionnaire items,
leaving a final sample of 2967 cases with 1460 complete and 1507 imputed.



/home/sina/Documents/Manuscripts/2026/Green_Spaces/src_codes/Analysis_paper/utils/psychometrics.py:111: UserWarning: 
IDS_I11: 10.5% missing responses.
  warnings.warn(
/home/sina/Documents/Manuscripts/2026/Green_Spaces/src_codes/Analysis_paper/utils/psychometrics.py:111: UserWarning: 
IDS_I12: 25.8% missing responses.
  warnings.warn(
/home/sina/Documents/Manuscripts/2026/Green_Spaces/src_codes/Analysis_paper/utils/psychometrics.py:111: UserWarning: 
IDS_I13: 13.8% missing responses.
  warnings.warn(
/home/sina/Documents/Manuscripts/2026/Green_Spaces/src_codes/Analysis_paper/utils/psychometrics.py:111: UserWarning: 
IDS_I14: 29.6% missing responses.
  warnings.warn(



Internal consistency
----------------------------------------
Number of participants : 2,967
Number of items        : 30
McDonald's Omega       : 0.876
Cronbach's Alpha       : 0.870




,IDS_EDAT,IDS_SUM,IDS_depression_severity,IDS_clinical_depression
SIC,,,,
1C0B901CC0,2014-05-21 06:51:56,11.000000,0,0
BEAAF0318F,2014-05-21 06:52:48,22.000000,1,0
C1FBC6AB69,2014-04-04 06:43:56,9.642857,0,0
6662694797,2014-06-19 00:00:00,13.928571,0,0
697C92C08C,2014-05-13 00:00:00,12.413793,0,0
...,...,...,...,...
6B3760F913,2011-10-18 11:36:31,10.000000,0,0
4B9D13E725,2011-10-13 08:00:00,34.357143,2,1
4E26F792AF,2011-08-22 14:22:42,18.000000,1,0


# GDS-15
## Geriatric Depression Scale 15-item

In [9]:
# --------------------------------------------------
# Read GDS-15 data
# --------------------------------------------------
df_gds15 = pd.read_excel(
    f"{BASE_PATH}/gds15.xlsx",
    index_col="SIC"
)
display(df_gds15.info())

# --------------------------------------------------
# GDS-15 items
# Five positively worded items are reverse scored so
# that higher scores consistently indicate greater
# depressive symptom severity
# --------------------------------------------------
subset = [f"GDS15_I{i:02d}" for i in range(1, 16)] # _I01:_I15

reverse_items = ["GDS15_I01", "GDS15_I05", "GDS15_I07", "GDS15_I11", "GDS15_I13"]

# Reverse score (0 ↔ 1)
df_gds15[reverse_items] = 1 - df_gds15[reverse_items]

# --------------------------------------------------
# Person-mean imputation
# Impute missing values using each participant's mean
# score when ≤20% of items (≤3 of 15) are missing
# --------------------------------------------------
df_gds15 = psychometrics.person_mean_imputation(
    df_gds15,
    subset_columns=subset,
    threshold=1/5,
)

assert not df_gds15[subset].isna().any().any()

# --------------------------------------------------
# Internal consistency
# --------------------------------------------------
reliability = psychometrics.reliability_test(df_gds15, subset)

# --------------------------------------------------
# Compute GDS-15 total score (range: 0–15)
# Higher scores indicate greater depressive symptoms
# --------------------------------------------------
df_gds15["GDS15_SUM"] = df_gds15[subset].sum(axis=1)

# --------------------------------------------------
# Depression severity
#
# 0:  0– 4 → No clinically relevant depressive symptoms
# 1:  5– 8 → Mild depression symptoms
# 2:  9–11 → Moderate depression symptoms
# 3: 12–15 → Severe depression symptoms
# --------------------------------------------------
df_gds15["GDS15_depression_severity"] = transform.categorize(
    df_gds15,
    "GDS15_SUM",
    bins=[0,5,9,12,np.inf],
    right=False,
)

# --------------------------------------------------
# Binary screening outcome
#
# 0: GDS-15 <  5 → No clinically relevant depressive symptoms
# 1: GDS-15 >= 5 → Screen-positive for depression
# --------------------------------------------------
df_gds15["GDS15_clinical_depression_risk"] = transform.binarize(
    df_gds15,
    "GDS15_SUM",
    threshold=5,
)

# --------------------------------------------------
# Keep required variables
# --------------------------------------------------
columns = [
    "GDS15_DATUM",
    "GDS15_SUM",
    "GDS15_depression_severity",
    "GDS15_clinical_depression_risk",
]
df_gds15 = df_gds15[columns]

display(df_gds15)

<class 'pandas.core.frame.DataFrame'>
Index: 2992 entries, 1C0B901CC0 to 37AD8E09F4
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   GDS15_DATUM         2992 non-null   datetime64[ns]
 1   GRUPPE              2992 non-null   object        
 2   GDS15_STARTZEIT     1624 non-null   datetime64[ns]
 3   GDS15_ENDZEIT       1624 non-null   datetime64[ns]
 4   GDS15_DQP_ID        2992 non-null   object        
 5   GDS15_UNTERSUCHER   1624 non-null   object        
 6   GDS15_TECH_CHECKED  2992 non-null   int64         
 7   GDS15_CURATED       2992 non-null   int64         
 8   GDS15_CHECKED       2992 non-null   int64         
 9   GDS15_I01           2974 non-null   float64       
 10  GDS15_I02           2959 non-null   float64       
 11  GDS15_I03           2971 non-null   float64       
 12  GDS15_I04           2975 non-null   float64       
 13  GDS15_I05           2963 non-null   fl

None


Missing Imputation
----------------------------------------
Of 2992 participants, 17 (0.6%) excluded for missing >20% of questionnaire items,
leaving a final sample of 2975 cases with 2808 complete and 167 imputed.

Internal consistency
----------------------------------------
Number of participants : 2,975
Number of items        : 15
McDonald's Omega       : 0.824
Cronbach's Alpha       : 0.812




,GDS15_DATUM,GDS15_SUM,GDS15_depression_severity,GDS15_clinical_depression_risk
SIC,,,,
1C0B901CC0,2014-05-21 06:51:56,3.0,0,0
BEAAF0318F,2014-05-21 06:52:48,5.0,1,1
C1FBC6AB69,2014-04-04 06:43:56,1.0,0,0
6662694797,2014-06-19 00:00:00,5.0,1,1
84A4A25FE9,2014-05-22 10:11:13,0.0,0,0
...,...,...,...,...
6B3760F913,2011-10-18 11:38:13,2.0,0,0
4B9D13E725,2011-10-13 08:00:00,5.0,1,1
4E26F792AF,2011-08-22 14:24:51,2.0,0,0


# Mental Health Data

In [10]:
data_mental = pd.concat([df_basis, df_cesd, df_gad7, df_swls, df_ids, df_gds15], join='outer', axis=1)
data_mental.shape, data_mental.columns

((10000, 24),
 Index(['TEILNEHMER_GESCHLECHT', 'ADULT_PROB_AGE', 'BASIS_EXAMINATION_YEAR',
        'BASIS_EXAMINATION_SEASON', 'CES_D_DATUM', 'CES_D_SUM',
        'CESD_depression_severity', 'CESD_clinical_depression', 'GAD7_DATUM',
        'GAD7_SUM', 'GAD7_anxiety_severity', 'GAD7_anxiety', 'SWLS_DATUM',
        'SWLS_SUM', 'SWLS_satisfaction_grade', 'SWLS_satisfaction', 'IDS_EDAT',
        'IDS_SUM', 'IDS_depression_severity', 'IDS_clinical_depression',
        'GDS15_DATUM', 'GDS15_SUM', 'GDS15_depression_severity',
        'GDS15_clinical_depression_risk'],
       dtype='object'))

## Save Mental Health Data to `mental_health_dataset_b.xlsx`

In [11]:
BASE_PATH = '../final_data/processed'
path = f'{BASE_PATH}/medical/basis/mental_health_dataset_b.xlsx'

##data_mental.to_excel(path, index=True)